### TF-IDF (Term Frequency-Inverse Document Frequency)

It's a statistical method used in natural language processing and information retrieval to evaluate how important a word is to a document in relation to a larger collection of documents. It combines two components : 

- Term Frequency (TF) that measures how often a word appears in a document. A higher frequency suggests greater importance and if a term appears frequently in a document, it seems relevant to the document's content.

- Inverse Document Frequency (IDF) reduces the weight of common words across multiples doucments while increasing the weight of rare words. If a term appears in fewer documents, it is more likely to be meaningful an specific. Example of common words : "the", "of", etc.

The final TF-IDF score = TF × IDF therefore gives greater weight to terms that are specific and representative of a document.
Documents and queries are then represented as TF-IDF vectors, and cosine similarity is used to measure their similarity and find the most relevant documents.


In [ ]:
import sys
import json
from pathlib import Path
import time
import numpy as np
import pandas as pd

# Resolve project root whether launched from project root or notebooks/
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(project_root))


In [ ]:
# Import TF-IDF functions
from src.retrieval.tfidf import fit_tfidf, retrieve_tfidf, map_indices_to_docids

# Import data functions
from src.data.load import load_all
from src.data.preprocess import add_content_field

# Import evaluation functions
from src.evaluation.evaluate import evaluate_run, evaluate_multi_k, adapt_ground_truth


In [ ]:
# Configuration
K_VALUES = [5, 10, 20, 50]
TEXT_FIELD = "content"

RAW_DIR = project_root / "data" / "raw"
PROCESSED_DIR = project_root / "data" / "processed"

# Set to None to run on the full corpus.
MAX_DOCS = 50000


### Data Loading and Preprocessing : 

The training documents and queries (preferably preprocessed) are loaded.
If the files do not exist, the raw files are reloaded and preprocessing is applied.

We load the ground truth and adapt it to the dictionary format.
If MAX_DOCS is active, we create an intelligent subset (all relevant + supplements).

In [ ]:
# Load documents/queries with content created in notebook 01 when available.
docs_path = PROCESSED_DIR / "docs_with_content.json"
queries_path = PROCESSED_DIR / "queries_train_with_content.json"

if docs_path.exists() and queries_path.exists():
    with open(docs_path, "r", encoding="utf-8") as f:
        docs = json.load(f)
    with open(queries_path, "r", encoding="utf-8") as f:
        queries = json.load(f)
else:
    docs_raw, train_queries_raw, _, _ = load_all(RAW_DIR)
    docs, queries = add_content_field(docs_raw, train_queries_raw, clean=True)

with open(RAW_DIR / "qgts_train.json", "r", encoding="utf-8") as f:
    qgts_train = json.load(f)

# Convert data ground-truth format to metrics format (qid -> [doc_ids]).
gt_full = adapt_ground_truth(qgts_train)
query_ids = [str(q["id"]) for q in queries]

# Optional subset for notebook speed while keeping relevant docs in the candidate set.
if MAX_DOCS is not None and len(docs) > MAX_DOCS:
    doc_by_id = {str(d["id"]): d for d in docs}
    relevant_ids = {
        doc_id
        for qid in query_ids
        for doc_id in gt_full.get(str(qid), [])
        if doc_id in doc_by_id
    }

    selected_ids = set(relevant_ids)
    if len(selected_ids) < MAX_DOCS:
        for d in docs:
            did = str(d["id"])
            if did in selected_ids:
                continue
            selected_ids.add(did)
            if len(selected_ids) >= MAX_DOCS:
                break

    docs = [d for d in docs if str(d["id"]) in selected_ids]
    gt = {
        str(qid): [doc_id for doc_id in gt_full.get(str(qid), []) if doc_id in selected_ids]
        for qid in query_ids
    }
else:
    gt = {str(qid): list(gt_full.get(str(qid), [])) for qid in query_ids}

print(f"Loaded {len(docs)} documents, {len(queries)} train queries, {len(gt)} ground-truth entries.")


### Fit with TF-IDF vectorizer :

We are going to fit the documents by using the TF-IDF vectorizer.

In [ ]:
# Fit TF-IDF
start_time = time.time()
vectorizer, doc_matrix = fit_tfidf(docs, text_field=TEXT_FIELD)
fit_time = time.time() - start_time

print(f"Fit time: {fit_time:.4f} seconds")
print("Doc matrix shape:", doc_matrix.shape)


### Retrieval and Evaluation : 

We will test the retrieval for k in [5, 10, 20, 50].
By doing so, we will measure the retrieval time and also compute the metrics.

In [ ]:
# Retrieval and evaluation for multiple k
k_values = [k for k in K_VALUES if k <= len(docs)]
if not k_values:
    raise ValueError("No valid k for current number of docs.")

results = []
for k in k_values:
    start_time = time.time()
    topk_indices, topk_scores = retrieve_tfidf(
        vectorizer, doc_matrix, queries, k, text_field=TEXT_FIELD
    )
    retrieve_time = time.time() - start_time

    pred_docids = map_indices_to_docids(topk_indices, docs)
    eval_results = evaluate_run(pred_docids, gt, query_ids, k)

    eval_results["fit_time_s"] = fit_time
    eval_results["retrieve_time_s"] = retrieve_time
    eval_results["avg_retrieve_time_ms"] = (retrieve_time / len(queries)) * 1000
    results.append(eval_results)

    print(f"\n--- Results for k={k} ---")
    print(f"Retrieve time: {retrieve_time:.4f} s (avg {eval_results["avg_retrieve_time_ms"]:.2f} ms/query)")
    print("Top-k indices shape:", topk_indices.shape)
    print("Top-k scores shape:", topk_scores.shape)
    print("Evaluate run:", eval_results)


### Summary table : 

A table that compile the previous results.

In [ ]:
# Results table
results_df = pd.DataFrame(results).sort_values("k").reset_index(drop=True)
display(results_df)

# Export results
output_dir = project_root / "outputs" / "runs"
output_dir.mkdir(parents=True, exist_ok=True)  # Create the folder, if necessary
results_df.to_csv(output_dir / "tfidf_results.csv", index=False)
print(f"Exported TF-IDF results to {output_dir / 'tfidf_results.csv'}")

### Consistent multi-k evaluation (single run at k_max)

In [ ]:
# Coherent multi-k evaluation from one retrieval run at max(k)
k_max = max(k_values)
topk_indices_max, topk_scores_max = retrieve_tfidf(
    vectorizer, doc_matrix, queries, k_max, text_field=TEXT_FIELD
)
pred_docids_max = map_indices_to_docids(topk_indices_max, docs)

multi_results = evaluate_multi_k(pred_docids_max, gt, query_ids, k_values)
print("\nMulti-k Results:")
display(multi_results)


Our actual results shows that TF-IDF is a strong and efficient baseline, however, it's not the best in ranking quality.

precision@k decreases while k increases, recall@k increases with k and mrr@k is quite stable.

### Top 100 preparation for queries_test (Kaggle submission):

- Load queries_test_with_content.json (or raw + preprocess if absent).
- Retrieve top 100 indices/scores with TF-IDF.
- Map to doc_ids.
- Create query_id -> [100 doc_ids] structure ready for src/kaggle/format.py.

In [ ]:
# Load queries_test
queries_test_path = PROCESSED_DIR / "queries_test_with_content.json"

if queries_test_path.exists():
    with open(queries_test_path, "r", encoding="utf-8") as f:
        queries_test = json.load(f)
else:
    # Fallback: load raw and preprocess (same as for train)
    _, _, test_queries_raw, _ = load_all(RAW_DIR)  # Ignore docs/train_queries/gts
    _, queries_test = add_content_field([], test_queries_raw, clean=True)  # Only queries

print(f"Loaded {len(queries_test)} test queries.")


# Retrieve top-100
k_test = 100
topk_indices_test, topk_scores_test = retrieve_tfidf(vectorizer, doc_matrix, queries_test, k_test, text_field=TEXT_FIELD)


# Map to doc_ids
pred_docids_test = map_indices_to_docids(topk_indices_test, docs)

# Get query_ids
query_ids_test = [str(q["id"]) for q in queries_test]


# Structure ready for format.py: dict query_id -> list[doc_ids] (optional, for debugging)
pred_dict = {q_id: doc_ids for q_id, doc_ids in zip(query_ids_test, pred_docids_test)}

print(f"Example for query_id {query_ids_test[0]}: top-5 doc_ids = {pred_docids_test[0][:5]}")

